In [1]:
from question_generator_agent.multiple_choince import MultipleChoiceQuestionGeneratorAgent
from question_generator_agent.subject_content import SLIDES_CONTENT

from dotenv import load_dotenv

load_dotenv()

True

In [5]:
agent = MultipleChoiceQuestionGeneratorAgent(
    subject_content=SLIDES_CONTENT,
    subject_name="Generative AI e Advanced Analytics",
)

In [6]:
question = agent.invoke({
    "question_topic": "Autoencoders Variacionais",
    "level": "difícil",
})

In [4]:
# question.model_dump()

In [7]:
import json

generated_question_text = json.dumps(question.model_dump(), ensure_ascii=False, indent=2)

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from question_generator_agent.models import QuestionQualityEvaluation
from question_generator_agent.prompts import QUESTION_QUALITY_EVALUATION_PROMPT

In [9]:
llm = ChatOpenAI(model="gpt-5-nano")

evaluation_prompt = ChatPromptTemplate.from_messages([
    ("system", QUESTION_QUALITY_EVALUATION_PROMPT),
])

evaluation_chain = (
    evaluation_prompt
    | llm.with_structured_output(QuestionQualityEvaluation)
)

# --- Execução ---
result = evaluation_chain.invoke({
    "generated_question": generated_question_text,
    "subject_name": "Generative AI e Advanced Analytics",
    "question_topic": "RAG",
    "question_type": "múltipla escolha",
    "level": "difícil",
})

# --- Acesso aos dados ---
print(f"Status: {result.global_status}")
print(f"Critérios: {len(result.criteria)}")
print(f"Problemas: {len(result.issues)}")

if result.issues:
    for issue in result.issues:
        print(f"\n[{issue.severity}] {issue.description}")
        print(f"  Critério: {issue.criterion_name}")
        print(f"  Trecho: {issue.problematic_excerpt}")
        print(f"  Sugestão: {issue.suggestion}")


Status: GlobalStatus.APPROVED_WITH_CAVEATS
Critérios: 8
Problemas: 2

[Severity.MEDIUM] O tema declarado não está alinhado com o conteúdo da questão (RAG vs VAEs).
  Critério: 8. ADERÊNCIA AO CONTEÚDO
  Trecho: Tema: RAG (indicação de tema declarado)
  Sugestão: Revisar o enunciado para alinhar o tema com VAEs ou, se o objetivo é abordar RAG, reformular a questão para discutir como RAG pode incorporar VAEs (ex.: usar VAEs como geradores de conteúdo em cenários de recuperação) e adicionar elementos que exijam comparação ou integração com RAG.

[Severity.MEDIUM] A avaliação de dificuldade classificada como 'difícil' não se reflete no conteúdo da questão, que testa principalmente conhecimento conceitual básico sobre VAEs sem exigir inferência ou derivação avançada.
  Critério: 7. ADEQUAÇÃO AO NÍVEL DE DIFICULDADE
  Trecho: Contextualização: Os autoencoders variacionais (VAEs) são modelos generativos probabilísticos... (descrição conceitual padrão)
  Sugestão: Aumentar o nível de exigência

In [13]:
evaluation_result_string = json.dumps(result.model_dump().get('issues'), ensure_ascii=False, indent=2)

In [14]:
print(evaluation_result_string)

[
  {
    "description": "O tema declarado não está alinhado com o conteúdo da questão (RAG vs VAEs).",
    "criterion_name": "8. ADERÊNCIA AO CONTEÚDO",
    "severity": "média",
    "problematic_excerpt": "Tema: RAG (indicação de tema declarado)",
    "suggestion": "Revisar o enunciado para alinhar o tema com VAEs ou, se o objetivo é abordar RAG, reformular a questão para discutir como RAG pode incorporar VAEs (ex.: usar VAEs como geradores de conteúdo em cenários de recuperação) e adicionar elementos que exijam comparação ou integração com RAG."
  },
  {
    "description": "A avaliação de dificuldade classificada como 'difícil' não se reflete no conteúdo da questão, que testa principalmente conhecimento conceitual básico sobre VAEs sem exigir inferência ou derivação avançada.",
    "criterion_name": "7. ADEQUAÇÃO AO NÍVEL DE DIFICULDADE",
    "severity": "média",
    "problematic_excerpt": "Contextualização: Os autoencoders variacionais (VAEs) são modelos generativos probabilísticos.

In [15]:
from question_generator_agent.models import CorrectedMultipleChoiceQuestion, serialize_issues, serialize_question
from question_generator_agent.prompts import QUESTION_CORRECTION_PROMPT

correction_prompt = ChatPromptTemplate.from_messages([
    ("system", QUESTION_CORRECTION_PROMPT),
])

correction_chain = (
    correction_prompt
    | llm.with_structured_output(CorrectedMultipleChoiceQuestion)
)

In [19]:
issues_str = serialize_issues(result.issues)
question_str = serialize_question(question)

In [21]:
print(issues_str)
print(question_str)

Problema 1:
  Descrição: O tema declarado não está alinhado com o conteúdo da questão (RAG vs VAEs).
  Critério afetado: 8. ADERÊNCIA AO CONTEÚDO
  Severidade: média
  Trecho problemático: Tema: RAG (indicação de tema declarado)
  Sugestão de correção: Revisar o enunciado para alinhar o tema com VAEs ou, se o objetivo é abordar RAG, reformular a questão para discutir como RAG pode incorporar VAEs (ex.: usar VAEs como geradores de conteúdo em cenários de recuperação) e adicionar elementos que exijam comparação ou integração com RAG.

Problema 2:
  Descrição: A avaliação de dificuldade classificada como 'difícil' não se reflete no conteúdo da questão, que testa principalmente conhecimento conceitual básico sobre VAEs sem exigir inferência ou derivação avançada.
  Critério afetado: 7. ADEQUAÇÃO AO NÍVEL DE DIFICULDADE
  Severidade: média
  Trecho problemático: Contextualização: Os autoencoders variacionais (VAEs) são modelos generativos probabilísticos... (descrição conceitual padrão)
  S

In [22]:
corrected = correction_chain.invoke({
    "generated_question": question_str,
    "issues": issues_str,
    "subject_name": "Generative AI e Advanced Analytics",
    "question_topic": "Retrieval-Augmented Generation (RAG)",
    "question_type": "múltipla escolha",
    "level": "dificil",
})

# --- Acesso aos dados ---
print(f"Questão: {corrected.question}")
print(f"Resposta: {corrected.answer}")
print(f"Gabarito: {corrected.correct_alternative}\n")

for alt in corrected.alternatives:
    marker = "✓" if alt.is_correct else "✗"
    print(f"  {marker} {alt.label}) {alt.text}")
    print(f"    → {alt.explanation}\n")

print("Correções aplicadas:")
for c in corrected.corrections_applied:
    print(f"  - {c}")

Questão: Contextualização: Retrieval-Augmented Generation (RAG) é uma abordagem que combina recuperação de documentos com geração de texto, de modo que o gerador tenha acesso a evidências relevantes para fundamentar as respostas. Em um cenário que integra VAEs com RAG, o gerador pode operar de forma probabilística, aprendendo a distribuir as saídas condicionais aos documentos recuperados fornecidos pela etapa de recuperação. Comando: Assinale a alternativa que melhor descreve uma característica fundamental da integração entre RAG e VAEs.
Resposta: Em um esquema RAG com VAEs, o gerador aprende uma distribuição probabilística do espaço latente e gera novas amostras condicionadas aos documentos recuperados fornecidos pelo componente de recuperação. (Treinamento geralmente envolve ELBO para aproximar p(x|z) e p(z)).
Gabarito: B

  ✗ A) Os VAEs aprendem uma representação determinística dos dados, garantindo que cada entrada seja codificada em um único vetor latente sem incerteza.
    → É in